# PIB 2026 — Forecast-aware Reinforcement Learning for Inventory Control

Local, reproducible version (refactored from the original Colab notebook).

**What this notebook does**
1. Loads M5 demand data (`sales_train_validation.csv`) if available, otherwise
   generates reproducible **synthetic** demand so it runs anywhere.
2. Trains a demand **forecaster** (LinearRegression or RandomForest).
3. Trains a **PPO** agent on a digital-twin inventory environment, *with* and
   *without* the forecast as a state feature.
4. Compares the two across items, seeds, and robustness scenarios
   (noisy forecast, carbon-aware), then summarises and plots the result.

**Key design choices (vs the original)**
- No Colab / Google Drive dependency; dynamic data path + synthetic fallback.
- **Lead time** on orders so the forecast is genuinely useful.
- **Normalised** observations + **service-level** reward for stable PPO training.
- De-duplicated into reusable functions; `boxplot(labels=...)` bug fixed.

See `EVALUATION.md` for the analysis of results.


## 0. Setup
Install dependencies if needed, then import.

In [ ]:
# If running in a fresh environment, uncomment:
# !pip install numpy pandas matplotlib scikit-learn stable-baselines3 gymnasium

from __future__ import annotations
import os, warnings
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

## 1. Data loading (dynamic path + synthetic fallback)
Looks for the real M5 CSV; if absent, builds reproducible seasonal demand.

In [ ]:
CANDIDATE_PATHS = [
    "sales_train_validation.csv",
    "data/sales_train_validation.csv",
    "PIB 2026/sales_train_validation.csv",
    os.path.expanduser("~/PIB 2026/sales_train_validation.csv"),
    "/content/drive/MyDrive/PIB 2026/sales_train_validation.csv",
]

def find_m5_csv():
    for p in CANDIDATE_PATHS:
        if Path(p).is_file():
            return p
    return None

def make_synthetic_sales(n_items=10, n_days=1200, seed=0):
    """Reproducible synthetic demand resembling M5 (id + d_1..d_T)."""
    rng = np.random.default_rng(seed)
    rows, t = [], np.arange(n_days)
    for i in range(n_items):
        base   = rng.uniform(8, 20)
        weekly = rng.uniform(3, 7) * np.sin(2*np.pi*t/7   + rng.uniform(0, 6.28))
        yearly = rng.uniform(2, 6) * np.sin(2*np.pi*t/365 + rng.uniform(0, 6.28))
        trend  = rng.uniform(-0.002, 0.004) * t
        noise  = rng.normal(0, 1.0, n_days)
        demand = np.clip(base + weekly + yearly + trend + noise, 0, None)
        rows.append(np.round(demand).astype(int))
    df = pd.DataFrame(rows, columns=[f"d_{j+1}" for j in range(n_days)])
    df.insert(0, "id", [f"ITEM_{k}" for k in range(n_items)])
    return df

def load_sales(n_items, seed=0):
    path = find_m5_csv()
    if path:
        print(f"[data] Loaded real M5 data from: {path}")
        return pd.read_csv(path), "m5"
    print("[data] M5 CSV not found -> using reproducible SYNTHETIC demand.")
    return make_synthetic_sales(n_items=max(n_items, 10), seed=seed), "synthetic"

def item_series(sales, idx):
    cols = [c for c in sales.columns if c.startswith("d_")]
    return sales.iloc[idx][cols].astype(float).values

sales, source = load_sales(n_items=3)
series0 = item_series(sales, 0)
plt.figure(figsize=(10, 3)); plt.plot(series0[:200])
plt.title(f"Demand time series (item 0, source={source})"); plt.show()

## 2. Forecast models
Lag features (last 7 days -> next day). Two models: Linear and RandomForest.

In [ ]:
def create_lag(series, lag=7):
    X, y = [], []
    for i in range(lag, len(series)):
        X.append(series[i-lag:i]); y.append(series[i])
    return np.array(X), np.array(y)

def build_forecaster(name):
    if name == "linear":
        return LinearRegression()
    if name == "rf":
        return RandomForestRegressor(n_estimators=60, max_depth=8, random_state=0, n_jobs=-1)
    raise ValueError(name)

def fit_forecaster(series, name, lag=7):
    X, y = create_lag(series, lag)
    split = int(0.8 * len(X))
    model = build_forecaster(name)
    model.fit(X[:split], y[:split])
    mae = mean_absolute_error(y[split:], model.predict(X[split:])) if split < len(X) else float("nan")
    return model, mae

for name in ["linear", "rf"]:
    _, mae = fit_forecaster(series0, name)
    print(f"{name:6s} forecast MAE = {mae:.3f}")

## 3. RL environment (digital twin)
Order-up-to control with **lead time** (orders arrive later -> forecast matters),
**normalised** observations, and a **service-level** reward (fill-rate minus
normalised holding, optional carbon penalty). Forecasts are precomputed per
episode so slow models (RF) stay fast.

In [ ]:
class InventoryEnv(gym.Env):
    def __init__(self, demand, model=None, lag=7, use_forecast=True,
                 forecast_noise=0.0, carbon=False, lead_time=2,
                 holding_cost=0.05, max_order_mult=3.0):
        super().__init__()
        self.demand = np.asarray(demand, dtype=np.float64)
        self.model, self.lag = model, lag
        self.use_forecast, self.forecast_noise, self.carbon = use_forecast, forecast_noise, carbon
        self.lead_time, self.holding_cost = int(lead_time), holding_cost
        self.demand_scale = max(1.0, float(np.mean(self.demand)))
        self.max_order = max_order_mult * self.demand_scale

        obs_dim = 4 if use_forecast else 3
        self.action_space = spaces.Box(0.0, 1.0, (1,), np.float32)
        self.observation_space = spaces.Box(-10.0, 50.0, (obs_dim,), np.float32)

        # precompute forecasts (deterministic in t) -> big speedup for RF
        self._forecast_cache = None
        if use_forecast and model is not None:
            idx = np.arange(self.lag, len(self.demand))
            X = np.stack([self.demand[i-self.lag:i] for i in idx])
            self._forecast_cache = np.zeros(len(self.demand))
            self._forecast_cache[idx] = model.predict(X)

    def _forecast(self):
        base = float(self._forecast_cache[self.t])
        if self.forecast_noise:
            base += float(np.random.normal(0, self.forecast_noise * self.demand_scale / 10.0))
        return base

    def _get_state(self):
        pipeline = float(sum(self.order_queue))
        s = [self.inventory, pipeline, self.backlog]
        if self.use_forecast:
            s.append(self._forecast())
        return np.array(s, dtype=np.float32) / self.demand_scale

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.t = self.lag
        self.inventory = self.demand_scale
        self.backlog = 0.0
        self.order_queue = deque([0.0]*self.lead_time, maxlen=self.lead_time)
        return self._get_state(), {}

    def step(self, action):
        order = float(np.clip(action[0], 0.0, 1.0)) * self.max_order
        arriving = self.order_queue.popleft() if self.order_queue else 0.0
        self.order_queue.append(order)
        self.inventory += arriving

        demand = self.demand[self.t]
        required = demand + self.backlog
        sales = min(self.inventory, required)
        self.inventory -= sales
        self.backlog = max(0.0, required - sales)

        fill_rate = sales/required if required > 0 else 1.0
        reward = fill_rate - self.holding_cost*(self.inventory/self.demand_scale)
        if self.carbon:
            reward -= 0.02*(order/self.demand_scale)

        self.t += 1
        done = self.t >= len(self.demand) - 1
        return self._get_state(), float(reward), done, False, {}

## 4. Train + evaluate helpers

In [ ]:
def evaluate_policy(env, model, steps=300):
    obs, _ = env.reset()
    rewards = []
    for _ in range(steps):
        a, _ = model.predict(obs, deterministic=True)
        obs, r, done, _, _ = env.step(a)
        rewards.append(r)
        if done:
            break
    return float(np.mean(rewards))

def run_experiment(series, forecaster="linear", forecast_noise=0.0, carbon=False,
                   seed=0, timesteps=7000, lag=7, lead_time=2):
    np.random.seed(seed)
    model_fc, mae = fit_forecaster(series, forecaster, lag)
    env_fc = InventoryEnv(series, model_fc, lag, True,  forecast_noise, carbon, lead_time)
    env_no = InventoryEnv(series, None,    lag, False, carbon=carbon, lead_time=lead_time)
    ppo_fc = PPO("MlpPolicy", env_fc, verbose=0, seed=seed); ppo_fc.learn(timesteps)
    ppo_no = PPO("MlpPolicy", env_no, verbose=0, seed=seed); ppo_no.learn(timesteps)
    return evaluate_policy(env_fc, ppo_fc), evaluate_policy(env_no, ppo_no), mae

## 5. Experiment grid
Items x forecast models x seeds x scenarios. Reduce `timesteps`/`seeds` for a
quick run; increase for sharper estimates.

In [ ]:
ITEMS, SEEDS, TIMESTEPS, LEAD_TIME = 3, 2, 7000, 2
FORECASTERS = ["linear", "rf"]
SCENARIOS = {
    "base":   dict(forecast_noise=0.0, carbon=False),
    "badfc":  dict(forecast_noise=1.5, carbon=False),
    "carbon": dict(forecast_noise=0.0, carbon=True),
}

records = []
for item in range(min(ITEMS, len(sales))):
    series = item_series(sales, item)
    for fc in FORECASTERS:
        for seed in range(SEEDS):
            for scen, kw in SCENARIOS.items():
                r_fc, r_no, mae = run_experiment(series, forecaster=fc, seed=seed,
                                                 timesteps=TIMESTEPS, lead_time=LEAD_TIME, **kw)
                records.append(dict(item=item, forecaster=fc, scenario=scen, seed=seed,
                                    reward_forecast=r_fc, reward_baseline=r_no,
                                    advantage=r_fc-r_no, forecast_mae=mae))
                print(f"item={item} fc={fc:6s} scen={scen:6s} seed={seed} "
                      f"| FC={r_fc:5.3f} NoFC={r_no:5.3f} adv={r_fc-r_no:+5.3f}")

df = pd.DataFrame(records)
df.to_csv("pib_rl_results.csv", index=False)
df.head()

## 6. Summary & statistics

In [ ]:
import math
print(df.groupby(["forecaster","scenario"])[["reward_forecast","reward_baseline","advantage"]]
        .agg(["mean","std"]).round(3))

adv = df["advantage"]
t = adv.mean()/(adv.std(ddof=1)/math.sqrt(len(adv))) if adv.std(ddof=1) > 0 else float("nan")
print(f"\nOVERALL forecast advantage: mean={adv.mean():+.3f} std={adv.std():.3f} "
      f"helped={100*(adv>0).mean():.0f}% of runs  t={t:.2f}")
print("\nPer-item advantage:")
print(df.groupby("item")["advantage"].agg(["mean","std","min","max"]).round(3))
print("\nForecast MAE by model:")
print(df.groupby("forecaster")["forecast_mae"].mean().round(3))

## 7. Visualisation (bug-fixed boxplot)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
groups, labels = [], []
for fc in FORECASTERS:
    for scen in ["base", "badfc", "carbon"]:
        groups.append(df[(df.forecaster==fc) & (df.scenario==scen)]["advantage"].values)
        labels.append(f"{fc}\n{scen}")
try:
    ax.boxplot(groups, tick_labels=labels)   # Matplotlib >= 3.9
except TypeError:
    ax.boxplot(groups, labels=labels)        # older Matplotlib
ax.axhline(0, color="red", ls="--", lw=1, label="no advantage")
ax.set_title("Forecast advantage (reward_FC - reward_baseline) by model x scenario")
ax.set_ylabel("advantage"); ax.legend(); plt.tight_layout(); plt.show()